# GraphTrustBench v7 — G6 GPU readouts
Runs `paper/scripts/v7_g6_readouts.py` on 4x L4 with vLLM from the competition wheelhouse. Inputs: the private dataset built by `v7_kaggle_package.py`. Outputs go to `/kaggle/working/<PART>`.


In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ.update(TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_FLAX='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1',
                  VLLM_WORKER_MULTIPROC_METHOD='spawn', VLLM_ENGINE_READY_TIMEOUT_S='1200', NCCL_DEBUG='WARN',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
# Wheelhouse install copied from the official getting-started notebook.
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/vllm-*.whl', recursive=True)[0]).parent
for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        try: os.symlink(w, tmp_whl / name)
        except OSError: shutil.copyfile(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
print('Wheelhouse installation complete.')

In [ ]:
PART = 'patch_comp'     # G6.2: 'patch_comp' (1,172) or 'patch_floor' (2 x 315); G6.3: 'settings:<comma list sharing one engine: para1,para2,para3,sampled | think | b8000 | m26 | e4>'
driver = r"""
import glob, hashlib, importlib.util, json, sys
# vLLM spawns workers that re-import this file; only the parent may run the setup and the readout.
if __name__ == "__main__":
    from pathlib import Path
    ds = Path(glob.glob('/kaggle/input/**/manifest.json', recursive=True)[0]).parent
    spec = importlib.util.spec_from_file_location('v7_kaggle_package', ds / 'paper__scripts__v7_kaggle_package.py')
    K = importlib.util.module_from_spec(spec); spec.loader.exec_module(K)
    art = Path('/kaggle/working/art')
    K.restore(ds, art)
    comp = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl', recursive=True)[0]).parent
    want = json.loads((art / 'paper/data/heldout_full_2026-11-07.json').read_text())['tasks_sha256']
    assert hashlib.sha256((comp / 'tasks.jsonl').read_bytes()).hexdigest() == want, 'tasks.jsonl differs from the frozen selection'
    MODELS = {'m26': 'gemma-4-26b-a4b-it', 'e4': 'gemma-4-e4b-it-qat-w4a16-ct'}
    name = MODELS.get(sys.argv[1].split(':')[-1].split(',')[0], 'gemma-4-31b-it-qat-w4a16-ct')
    model = Path(glob.glob(f'/kaggle/input/models/**/{name}/**/config.json', recursive=True)[0]).parent
    print('inputs', ds, 'comp', comp, 'model', model, flush=True)
    sys.path.insert(0, str(art / 'paper/scripts'))
    import v7_g6_readouts as G
    out = Path('/kaggle/working/' + sys.argv[1])
    if sys.argv[1] in ('patch_comp', 'patch_floor'):
        generate, encode, decode, _ = G.vllm_generate(model, out, max_tokens=4096)
        G.run_patch_comp(art, comp, generate, encode, decode, out, which=('main',) if sys.argv[1] == 'patch_comp' else ('floor',))
    elif sys.argv[1].startswith('settings:'):
        cfg = json.loads((model / 'generation_config.json').read_text())
        print('generation_config', cfg, flush=True)
        G.run_settings(sys.argv[1].split(':')[1].split(','), art, comp, model, Path('/kaggle/working/settings'), gen_config=cfg)
    else:
        raise SystemExit('unknown PART ' + sys.argv[1])
"""
Path('/kaggle/working/driver.py').write_text(driver)
proc = subprocess.run([sys.executable, '/kaggle/working/driver.py', PART])
print('exit', proc.returncode)
assert proc.returncode == 0